# Adding and Removing Data

## About the Data
In this notebook, we will be working with earthquake data from September 18, 2018 - October 13, 2018 (obtained from the US Geological Survey (USGS) using the [USGS API](https://earthquake.usgs.gov/fdsnws/event/1/))

## Setup
We will be working with the `data/earthquakes.csv` file again, so we need to handle our imports and read it in.

**한글 요약**

- 열 추가: `df['새열'] = 값` (제일 간단) / `df.assign(새열=값, ...)` (여러 개 한 번에)
- 표 이어붙이기: `pd.concat([표1, 표2])` → axis=0 위아래(기본), axis=1 좌우
- 삭제: `del df['열']` / `df.pop('열')` (빼내서 보관) / `df.drop(...)` (행도 열도)
- 중요: pandas 명령어 대부분은 원본을 안 바꾸고 새 결과를 돌려줌
  → 원본을 바꾸려면 다시 대입 `df = df.drop(...)` 하거나 `inplace=True`
- lambda(람다) 함수가 처음 나옴. `lambda x: 결과` = 이름 없는 한 줄 함수. assign 안에서 x는 "지금 만들어지고 있는 표"

In [ ]:
import pandas as pd

# usecols=[...] → 26개 열 전부 말고 이 7개 열만 읽어와 (필요한 것만 읽으면 가볍고 보기 편함)
df = pd.read_csv(
    'data/earthquakes.csv',
    usecols=['time', 'title', 'place', 'magType', 'mag', 'alert', 'tsunami']
)


## Creating new data
### Adding new columns
New columns get added to the right of the original columns and can be a single value, which will be **broadcast** along the rows of the dataframe:

In [ ]:
# df['새열이름'] = 값 → 새 열 만들기. 없는 열 이름에 값을 넣으면 맨 오른쪽에 새 열이 생김
# 값 하나('USGS API')만 넣으면 모든 행에 똑같이 채워짐 → 이걸 broadcast(브로드캐스트, 퍼뜨리기)라고 함
df['source'] = 'USGS API'
df.head()


,alert,mag,magType,place,time,title,tsunami,source
0,NaN,1.35,ml,"9km NE of Aguanga, CA",1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,USGS API
1,NaN,1.29,ml,"9km NE of Aguanga, CA",1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,USGS API
2,NaN,3.42,ml,"8km NE of Aguanga, CA",1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,USGS API
3,NaN,0.44,ml,"9km NE of Aguanga, CA",1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,USGS API
4,NaN,2.16,md,"10km NW of Avenal, CA",1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,USGS API


...or a Boolean mask:

In [ ]:
# df.mag < 0 → 행마다 "규모가 음수인가?" True/False (불리언 마스크)
# 그걸 그대로 새 열 'mag_negative'에 넣음 → True/False 열이 생김
df['mag_negative'] = df.mag < 0
df.head()


,alert,mag,magType,place,time,title,tsunami,source,mag_negative
0,NaN,1.35,ml,"9km NE of Aguanga, CA",1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,USGS API,False
1,NaN,1.29,ml,"9km NE of Aguanga, CA",1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,USGS API,False
2,NaN,3.42,ml,"8km NE of Aguanga, CA",1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,USGS API,False
3,NaN,0.44,ml,"9km NE of Aguanga, CA",1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,USGS API,False
4,NaN,2.16,md,"10km NW of Avenal, CA",1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,USGS API,False


#### Adding the `parsed_place` column
We have an entity recognition problem on our hands with the `place` column. There are several entities that have multiple names in the data (e.g., CA and California, NV and Nevada).

In [ ]:
# place 열이 '9km NE of Aguanga, CA' 처럼 되어 있는데, 콤마 뒤의 지역 이름만 뽑아보려는 거임
# .str.extract(r', (.*$)') → 정규표현식으로 ", " 뒤에 오는 것(.*)을 끝($)까지 뽑아냄. 괄호 () 안이 뽑히는 부분
# [0] → extract 결과는 데이터프레임이라 첫 번째 열만 꺼냄
# .sort_values() → 알파벳순 정렬 / .unique() → 중복 제거
# 결과를 보면 'CA'와 'California', 'NV'와 'Nevada' 처럼 같은 곳인데 이름이 다른 게 섞여 있음 → 정리 필요!
df.place.str.extract(r', (.*$)')[0].sort_values().unique()


array(['Afghanistan', 'Alaska', 'Argentina', 'Arizona', 'Arkansas',
       'Australia', 'Azerbaijan', 'B.C., MX', 'Barbuda', 'Bolivia',
       'Bonaire, Saint Eustatius and Saba ', 'British Virgin Islands',
       'Burma', 'CA', 'California', 'Canada', 'Chile', 'China',
       'Christmas Island', 'Colombia', 'Colorado', 'Costa Rica',
       'Dominican Republic', 'East Timor', 'Ecuador', 'Ecuador region',
       'El Salvador', 'Fiji', 'Greece', 'Greenland', 'Guam', 'Guatemala',
       'Haiti', 'Hawaii', 'Honduras', 'Idaho', 'Illinois', 'India',
       'Indonesia', 'Iran', 'Iraq', 'Italy', 'Jamaica', 'Japan', 'Kansas',
       'Kentucky', 'Kyrgyzstan', 'Martinique', 'Mauritius', 'Mayotte',
       'Mexico', 'Missouri', 'Montana', 'NV', 'Nevada', 'New Caledonia',
       'New Hampshire', 'New Mexico', 'New Zealand', 'Nicaragua',
       'North Carolina', 'Northern Mariana Islands', 'Oklahoma', 'Oregon',
       'Pakistan', 'Papua New Guinea', 'Peru', 'Philippines',
       'Puerto Rico', 'Roman

Replace parts of the `place` names to fit our needs:

In [ ]:
# .str.replace(찾을것, 바꿀것) 을 계속 이어 붙여서 place를 깨끗하게 정리하는 거임
# regex=True → 찾을 것이 정규표현식이라는 뜻. 정규표현식 기호: .* = 아무 글자 여러 개, $ = 끝, ^ = 시작
# 정규표현식은 어려우니 "이런 규칙으로 문자열을 찾아 바꾼다" 정도로만 이해해도 됨
# 예시로 '9km NE of Aguanga, CA' 가 어떻게 바뀌는지 따라가보면:
df['parsed_place'] = df.place.str.replace(
    r'.* of ', '', regex=True # remove anything saying <something> of <something> → 'Aguanga, CA'
).str.replace(
    'the ', '' # remove "the " → 'the ' 글자 삭제 (여기선 해당 없음)
).str.replace(
    r'CA$', 'California', regex=True # fix California → 'Aguanga, California'
).str.replace(
    r'NV$', 'Nevada', regex=True # fix Nevada → 끝이 NV면 Nevada로
).str.replace(
    r'MX$', 'Mexico', regex=True # fix Mexico → 끝이 MX면 Mexico로
).str.replace(
    r' region$', '', regex=True # chop off endings with " region" → 끝의 ' region' 삭제
).str.replace(
    'northern ', '' # remove "northern " → 'northern ' 삭제
).str.replace(
    'Fiji Islands', 'Fiji' # line up the Fiji places → Fiji Islands를 Fiji로 통일
).str.replace(
    r'^.*, ', '', regex=True # remove anything else extraneous from the beginning → 콤마 앞 전부 삭제: 'California'
).str.strip() # remove any extra spaces → 앞뒤 공백 제거
# 이렇게 정리한 결과를 새 열 'parsed_place'에 저장 (parse = 분석해서 뽑아내다)


Now we can use a single name to get all earthquakes for that place (although this still isn't perfect):

In [ ]:
# 정리된 parsed_place의 서로 다른 값들. 이제 CA/California가 'California' 하나로 합쳐짐
df.parsed_place.sort_values().unique()


array(['Afghanistan', 'Alaska', 'Argentina', 'Arizona', 'Arkansas',
       'Ascension Island', 'Australia', 'Azerbaijan', 'Balleny Islands',
       'Barbuda', 'Bolivia', 'British Virgin Islands', 'Burma',
       'California', 'Canada', 'Carlsberg Ridge',
       'Central East Pacific Rise', 'Central Mid-Atlantic Ridge', 'Chile',
       'China', 'Christmas Island', 'Colombia', 'Colorado', 'Costa Rica',
       'Dominican Republic', 'East Timor', 'Ecuador', 'El Salvador',
       'Fiji', 'Greece', 'Greenland', 'Guam', 'Guatemala', 'Haiti',
       'Hawaii', 'Honduras', 'Idaho', 'Illinois', 'India',
       'Indian Ocean Triple Junction', 'Indonesia', 'Iran', 'Iraq',
       'Italy', 'Jamaica', 'Japan', 'Kansas', 'Kentucky',
       'Kermadec Islands', 'Kuril Islands', 'Kyrgyzstan', 'Martinique',
       'Mauritius', 'Mayotte', 'Mexico', 'Mid-Indian Ridge', 'Missouri',
       'Montana', 'Nevada', 'New Caledonia', 'New Hampshire',
       'New Mexico', 'New Zealand', 'Nicaragua', 'North Carolina',


#### Using the `assign()` method to create columns
To create many columns at once or update existing columns, we can use `assign()`:

In [ ]:
# .assign(새열=값, 새열2=값2, ...) → 열 여러 개를 한 번에 만듦. 원본 df는 안 바뀌고 새 데이터프레임을 돌려줌
# in_ca → parsed_place가 'California'로 끝나나? (.str.endswith) True/False
# in_alaska → 'Alaska'로 끝나나?
# .sample(5, random_state=0) → 랜덤으로 5행 뽑기. random_state=0은 랜덤 고정(seed랑 같은 역할)
df.assign(
    in_ca=df.parsed_place.str.endswith('California'),
    in_alaska=df.parsed_place.str.endswith('Alaska')
).sample(5, random_state=0)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place,in_ca,in_alaska
7207,NaN,4.80,mwr,"73km SSW of Masachapa, Nicaragua",1537749595210,"M 4.8 - 73km SSW of Masachapa, Nicaragua",0,USGS API,False,Nicaragua,False,False
4755,NaN,1.09,ml,"28km NNW of Packwood, Washington",1538227540460,"M 1.1 - 28km NNW of Packwood, Washington",0,USGS API,False,Washington,False,False
4595,NaN,1.80,ml,"77km SSW of Kaktovik, Alaska",1538259609862,"M 1.8 - 77km SSW of Kaktovik, Alaska",0,USGS API,False,Alaska,False,True
3566,NaN,1.50,ml,"102km NW of Arctic Village, Alaska",1538464751822,"M 1.5 - 102km NW of Arctic Village, Alaska",0,USGS API,False,Alaska,False,True
2182,NaN,0.90,ml,"26km ENE of Pine Valley, CA",1538801713880,"M 0.9 - 26km ENE of Pine Valley, CA",0,USGS API,False,California,True,False


With the use of `lambda` functions, the `assign()` method becomes even more powerful. **Lambda functions** are anonymous functions usually defined in one line and for single use. The `assign()` method passes the entire dataframe into the `lambda` function as `x`; from there, we can select the columns `in_ca` and `in_alaska`, which are being created in that same call to `assign()`. Here, we use a `lambda` function to create a new column, `neither`, which tells if the earthquake was neither in Alaska nor California:

In [ ]:
# lambda(람다) 함수: 이름 없는 한 줄짜리 함수. lambda x: 결과 형태
#   def f(x): return 결과   ← 이걸 한 줄로 줄인 거임
# assign 안에서 lambda x: 를 쓰면 x에 '지금 만들어지고 있는 데이터프레임'이 통째로 들어옴
# 그래서 같은 assign 안에서 방금 만든 in_ca, in_alaska 열을 x.in_ca, x.in_alaska로 바로 쓸 수 있음!
# (lambda 없이 df.in_ca 라고 쓰면 df에는 아직 in_ca가 없어서 에러남)
# ~ 는 NOT(반대). ~x.in_ca & ~x.in_alaska → 캘리포니아도 아니고 알래스카도 아닌 지진 → neither(둘 다 아님)
df.assign(
    in_ca=df.parsed_place == 'California',
    in_alaska=df.parsed_place == 'Alaska',
    neither=lambda x: ~x.in_ca & ~x.in_alaska
).sample(5, random_state=0)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place,in_ca,in_alaska,neither
7207,NaN,4.80,mwr,"73km SSW of Masachapa, Nicaragua",1537749595210,"M 4.8 - 73km SSW of Masachapa, Nicaragua",0,USGS API,False,Nicaragua,False,False,True
4755,NaN,1.09,ml,"28km NNW of Packwood, Washington",1538227540460,"M 1.1 - 28km NNW of Packwood, Washington",0,USGS API,False,Washington,False,False,True
4595,NaN,1.80,ml,"77km SSW of Kaktovik, Alaska",1538259609862,"M 1.8 - 77km SSW of Kaktovik, Alaska",0,USGS API,False,Alaska,False,True,False
3566,NaN,1.50,ml,"102km NW of Arctic Village, Alaska",1538464751822,"M 1.5 - 102km NW of Arctic Village, Alaska",0,USGS API,False,Alaska,False,True,False
2182,NaN,0.90,ml,"26km ENE of Pine Valley, CA",1538801713880,"M 0.9 - 26km ENE of Pine Valley, CA",0,USGS API,False,California,True,False,False


#### Concatenation
Say we were working with two separate dataframes, one with earthquakes accompanied by tsunamis and the other with earthquakes without tsunamis. If we wanted to look at earthquakes as a whole, we would want to concatenate the dataframes into a single one:

In [ ]:
# 쓰나미 있는 지진 / 없는 지진으로 데이터프레임 두 개로 쪼갬 (필터링)
tsunami = df[df.tsunami == 1]
no_tsunami = df[df.tsunami == 0]

# 각각 크기 확인. 61개 + 9271개 = 9332개 (원래 전체 개수)
tsunami.shape, no_tsunami.shape


((61, 10), (9271, 10))

Concatenating along the row axis (`axis=0`) is equivalent to appending to the bottom. By concatenating our earthquakes with tsunamis and those without tsunamis, we get the full earthquake data set back:

In [ ]:
# pd.concat([df1, df2]) → 데이터프레임들을 이어붙임 (concat = concatenate = 연결)
# 기본은 위아래로(행 방향) 붙임 → 61 + 9271 = 9332행. 원래 데이터로 복원됨
pd.concat([tsunami, no_tsunami]).shape


(9332, 10)

Note that the previous result is equivalent to running the `append()` method of the dataframe:

In [ ]:
# .append() → concat과 같은 결과. tsunami 아래에 no_tsunami를 붙임
# ※ 최신 pandas(2.0 이상)에서는 append()가 삭제되어 에러남! 그냥 위의 pd.concat()을 쓰면 됨
tsunami.append(no_tsunami).shape


(9332, 10)

We have been working with a subset of the columns from the CSV file, but suppose that now we want to get some of the columns we ignored when we read in the data. Since we have added new columns in this notebook, we won't want to read in the file and perform those operations again. Instead, we will concatenate along the columns (`axis=1`) to add back what we are missing:

In [ ]:
# 처음에 안 읽었던 열들(tz, felt, ids)을 따로 읽어옴
additional_columns = pd.read_csv(
    'data/earthquakes.csv', usecols=['tz', 'felt', 'ids']
)
# axis=1 → 이번엔 옆으로(열 방향) 붙임. axis=0은 위아래(행), axis=1은 좌우(열)
# 인덱스(0, 1)가 서로 같으니까 같은 행끼리 옆으로 딱 붙음
pd.concat([df.head(2), additional_columns.head(2)], axis=1)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place,felt,ids,tz
0,NaN,1.35,ml,"9km NE of Aguanga, CA",1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,USGS API,False,California,NaN,",ci37389218,",-480.0
1,NaN,1.29,ml,"9km NE of Aguanga, CA",1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,USGS API,False,California,NaN,",ci37389202,",-480.0


Notice what happens if the index doesn't align though:

In [ ]:
# 이번엔 index_col='time' → time 열을 인덱스로 써서 읽음. 그러면 인덱스가 0,1이 아니라 큰 숫자(시간)가 됨
additional_columns = pd.read_csv(
    'data/earthquakes.csv', usecols=['tz', 'felt', 'ids', 'time'], index_col='time'
)
# df의 인덱스는 0, 1 / additional_columns의 인덱스는 1539475168010, ... → 안 맞음!
# 인덱스가 다르면 같은 행으로 안 보고 따로따로 붙여서, 빈칸(NaN)이 잔뜩 생김. 2행 + 2행 = 4행이 됨
# → 옆으로 붙일 때는 인덱스가 맞는지 꼭 확인!
pd.concat([df.head(2), additional_columns.head(2)], axis=1)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place,felt,ids,tz
0,NaN,1.35,ml,"9km NE of Aguanga, CA",1.539475e+12,"M 1.4 - 9km NE of Aguanga, CA",0.0,USGS API,False,California,NaN,NaN,NaN
1,NaN,1.29,ml,"9km NE of Aguanga, CA",1.539475e+12,"M 1.3 - 9km NE of Aguanga, CA",0.0,USGS API,False,California,NaN,NaN,NaN
1539475129610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",ci37389202,",-480.0
1539475168010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,",ci37389218,",-480.0


If the index doesn't align, we can align it before attempting the concatentation, which we will discuss in chapter 3.

Say we want to join the `tsunami` and `no_tsunami` dataframes, but the `no_tsunami` dataframe has an additional column. The `join` parameter specifies how to handle any overlap in column names (when appending to the bottom) or in row names (when concatenating to the left/right). By default, this is `outer`, so we keep everything; however, if we use `inner`, we will only keep what is in common:

In [ ]:
# no_tsunami 쪽에만 'type' 열을 추가해서(assign) 열 구성이 다른 두 표를 위아래로 붙여봄
# join='inner' → 양쪽에 '공통으로 있는' 열만 남김 (type 열은 한쪽에만 있으니 사라짐)
# 기본값은 join='outer' → 전부 남기고 없는 쪽은 NaN
pd.concat(
    [tsunami.head(2), no_tsunami.head(2).assign(type='earthquake')], join='inner'
)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place
36,NaN,5.00,mww,"165km NNW of Flying Fish Cove, Christmas Island",1539459504090,"M 5.0 - 165km NNW of Flying Fish Cove, Christm...",1,USGS API,False,Christmas Island
118,green,6.70,mww,"262km NW of Ozernovskiy, Russia",1539429023560,"M 6.7 - 262km NW of Ozernovskiy, Russia",1,USGS API,False,Russia
0,NaN,1.35,ml,"9km NE of Aguanga, CA",1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,USGS API,False,California
1,NaN,1.29,ml,"9km NE of Aguanga, CA",1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,USGS API,False,California


In addition, we use `ignore_index`, since the index doesn't mean anything for us here. This gives us sequential values instead of what we had in the previous result:

In [ ]:
# ignore_index=True → 원래 인덱스(36, 118, 0, 1) 버리고 0,1,2,3으로 새로 번호 매김
# 인덱스가 딱히 의미 없을 때 씀
pd.concat(
    [tsunami.head(2), no_tsunami.head(2).assign(type='earthquake')], join='inner', ignore_index=True
)


,alert,mag,magType,place,time,title,tsunami,source,mag_negative,parsed_place
0,NaN,5.00,mww,"165km NNW of Flying Fish Cove, Christmas Island",1539459504090,"M 5.0 - 165km NNW of Flying Fish Cove, Christm...",1,USGS API,False,Christmas Island
1,green,6.70,mww,"262km NW of Ozernovskiy, Russia",1539429023560,"M 6.7 - 262km NW of Ozernovskiy, Russia",1,USGS API,False,Russia
2,NaN,1.35,ml,"9km NE of Aguanga, CA",1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,USGS API,False,California
3,NaN,1.29,ml,"9km NE of Aguanga, CA",1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,USGS API,False,California


## Deleting Unwanted Data
Columns can be deleted using dictionary syntax with `del`:

In [ ]:
# del df['열이름'] → 열 삭제. 딕셔너리에서 키 지우는 것과 같은 문법
del df['source']
# 열 목록 확인 → source가 사라짐
df.columns


Index(['alert', 'mag', 'magType', 'place', 'time', 'title', 'tsunami',
       'mag_negative', 'parsed_place'],
      dtype='object')

If we don't know whether the column exists, we should use a `try`/`except` block:

In [ ]:
# 이미 지운 열을 또 지우려 하면 KeyError(그런 키 없음) 에러가 남
# try: ~ except 에러종류: ~ → "일단 해보고(try), 에러 나면(except) 이렇게 해" 라는 에러 처리 문법
# 프로그램이 멈추지 않고 'not there anymore'만 출력하고 넘어감
try:
    del df['source']
except KeyError:
    # handle the error here
    print('not there anymore')


not there anymore


We can also use `pop()`. This will allow us to use the series we remove later. Note there will be an error if the key doesn't exist, so we can also use a `try`/`except` here:

In [ ]:
# .pop('열이름') → 열을 빼내면서 그 열을 돌려줌. 지우긴 하는데 버리지 않고 변수에 저장해두는 거임
# del은 그냥 삭제, pop은 삭제 + 꺼내기 (리스트의 pop과 같음)
mag_negative = df.pop('mag_negative')
df.columns


Index(['alert', 'mag', 'magType', 'place', 'time', 'title', 'tsunami',
       'parsed_place'],
      dtype='object')

Notice we have a mask in `mag_negative` now:

In [ ]:
# 빼낸 mag_negative는 True/False 시리즈(마스크). 음수 규모가 491개
mag_negative.value_counts()


False    8841
True      491
Name: mag_negative, dtype: int64

Now, we can use `mag_negative` to filter our data:

In [ ]:
# 빼낸 마스크로 필터링. True인 행(규모가 음수인 지진)만 나옴
df[mag_negative].head()


,alert,mag,magType,place,time,title,tsunami,parsed_place
39,NaN,-0.10,ml,"6km NW of Lemmon Valley, Nevada",1539458844506,"M -0.1 - 6km NW of Lemmon Valley, Nevada",0,Nevada
49,NaN,-0.10,ml,"6km NW of Lemmon Valley, Nevada",1539455017464,"M -0.1 - 6km NW of Lemmon Valley, Nevada",0,Nevada
135,NaN,-0.40,ml,"10km SSE of Beatty, Nevada",1539422175717,"M -0.4 - 10km SSE of Beatty, Nevada",0,Nevada
161,NaN,-0.02,md,"20km SSE of Ronan, Montana",1539412475360,"M -0.0 - 20km SSE of Ronan, Montana",0,Montana
198,NaN,-0.20,ml,"60km N of Pahrump, Nevada",1539398340822,"M -0.2 - 60km N of Pahrump, Nevada",0,Nevada


### Using the `drop()` method
We can drop rows by passing a list of indices to the `drop()` method. Notice in the following example that when asking for the first 2 rows with `head()` we get the 3rd and 4th rows because we dropped the original first 2 with `drop([0, 1])`:

In [ ]:
# .drop([0, 1]) → 인덱스 0번, 1번 행을 삭제
# 그래서 head(2)를 해도 2, 3번 행이 나옴
# 주의: drop은 원본을 안 바꾸고 삭제된 '새 데이터프레임'을 돌려줌. df는 그대로임
df.drop([0, 1]).head(2)


,alert,mag,magType,place,time,title,tsunami,parsed_place
2,NaN,3.42,ml,"8km NE of Aguanga, CA",1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,California
3,NaN,0.44,ml,"9km NE of Aguanga, CA",1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,California


The `drop()` method drops along the row axis by default. If we pass in a list of columns with the `columns` argument, we can delete columns:

In [ ]:
# 남길 열 5개(alert, mag, title, time, tsunami)를 빼고 나머지 열 이름을 리스트로 만듦
# col not in [...] → 이 리스트에 없는 열만
cols_to_drop = [
    col for col in df.columns
    if col not in ['alert', 'mag', 'title', 'time', 'tsunami']
]
# .drop(columns=리스트) → 그 열들 삭제. drop은 기본이 행 삭제라서 열 지울 땐 columns= 라고 써줘야 함
df.drop(columns=cols_to_drop).head()


,alert,mag,time,title,tsunami
0,NaN,1.35,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0
1,NaN,1.29,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0
2,NaN,3.42,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0
3,NaN,0.44,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0
4,NaN,2.16,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0


We also have the option of using `axis=1`:

In [ ]:
# drop(columns=...) 와 drop(..., axis=1) 은 같은 뜻 (axis=1 = 열 방향)
df.drop(columns=cols_to_drop).equals(
    df.drop(cols_to_drop, axis=1)
)


True

By default, `drop()`, along with the majority of `DataFrame` methods, will return a new `DataFrame` object. If we just want to change the one we are working with, we can pass `inplace=True`. This should be used with care:

In [ ]:
# inplace=True → 새 데이터프레임을 돌려주는 대신 원본 df를 직접 바꿈
# 편하지만 되돌릴 수 없으니 조심해서 써야 함
df.drop(columns=cols_to_drop, inplace=True)
# 이제 df 자체가 5개 열만 남음
df.head()


,alert,mag,time,title,tsunami
0,NaN,1.35,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0
1,NaN,1.29,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0
2,NaN,3.42,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0
3,NaN,0.44,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0
4,NaN,2.16,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0


<hr>

<div style="overflow: hidden; margin-bottom: 10px;">
    <div style="float: left;">
        <a href="./5-subsetting_data.ipynb">
            <button>&#8592; Previous Notebook</button>
        </a>
    </div>
    <div style="float: right;">
        <a href="../../solutions/ch_02/solutions.ipynb">
            <button>Solutions</button>
        </a>
        <a href="../ch_03/1-wide_vs_long.ipynb">
            <button>Chapter 3 &#8594;</button>
        </a>
    </div>
</div>
<hr>